In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

C:\Users\YOGA\anaconda3\envs\tfcpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. LOAD DATA
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
df = pd.DataFrame(dataset["train"])

X = df["text"]
y = df["label"]

label_names = ["Negative", "Neutral", "Positive"]

# 2. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 1),
    min_df=3,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

ReadTimeout: The read operation timed out

In [ ]:
# # 4. MODELS
# models = {
#     "Logistic Regression": LogisticRegression(
#         max_iter=1000,
#         class_weight="balanced",
#     ),
    
#     "SVM": LinearSVC(
#         class_weight="balanced"
#     ),
    
#     "CatBoost": CatBoostClassifier(
#         iterations=100,
#         learning_rate=0.1,
#         depth=4,
#         loss_function="MultiClass",
#         verbose=0,
#         random_seed=42,
#         allow_writing_files=False
#     )
# }
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    
    "SVM": LinearSVC(
        class_weight="balanced"
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        objective="multi:softmax",  # zwraca klasy 0/1/2
        num_class=3,
        random_state=42,
        verbosity=0,
        n_jobs=-1
    )
}

# 5. TRAINING + EVALUATION

results = []
predictions = {}

for name, model in models.items():
    print(f"\n==================== {name} ====================")
    
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    y_pred = np.array(y_pred).ravel()
    
    predictions[name] = y_pred
    
    acc = accuracy_score(y_test, y_pred)
    f1_weighted = f1_score(y_test, y_pred, average="weighted")
    f1_macro = f1_score(y_test, y_pred, average="macro")
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1 weighted": f1_weighted,
        "F1 macro": f1_macro
    })
    
    print("Accuracy:", round(acc, 4))
    print("F1 weighted:", round(f1_weighted, 4))
    print("F1 macro:", round(f1_macro, 4))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=label_names))

# 6. RESULTS TABLE

results_df = pd.DataFrame(results)
print("\nFinal results:")
print(results_df)


# 7. BARPLOT COMPARISON
results_plot = results_df.set_index("Model")

fig, ax = plt.subplots(figsize=(10, 5))

colors = ["#4C72B0", "#55A868", "#C44E52"]  # niebieski, zielony, czerwony

results_plot[["Accuracy", "F1 weighted", "F1 macro"]].plot(
    kind="bar",
    ax=ax,
    color=colors
)

ax.set_title("Comparison of Classical ML Models", fontsize=14)
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_xticklabels(results_plot.index, rotation=0)

# siatka pozioma
ax.yaxis.grid(True, linestyle="--", alpha=0.6)

# legenda
ax.legend(loc="lower right")

# wartości nad słupkami
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()




In [ ]:
# 8. CONFUSION MATRICES
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(6, 5))
    
    plt.imshow(cm, cmap="Blues")
    plt.title(f"Confusion Matrix - {name}", fontsize=14)
    plt.colorbar()
    
    plt.xticks(np.arange(3), label_names)
    plt.yticks(np.arange(3), label_names)
    
    # wartości w komórkach
    for i in range(3):
        for j in range(3):
            plt.text(
                j, i, cm[i, j],
                ha="center",
                va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black",
                fontsize=12
            )
    
    plt.xlabel("Predicted label", fontsize=12)
    plt.ylabel("True label", fontsize=12)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 9. MOST IMPORTANT WORDS - LOGISTIC REGRESSION

feature_names = vectorizer.get_feature_names_out()

logreg_model = models["Logistic Regression"]
coef = logreg_model.coef_

for i, label in enumerate(label_names):
    top_idx = np.argsort(coef[i])[-15:]
    top_words = feature_names[top_idx]
    top_values = coef[i][top_idx]

    plt.figure(figsize=(8, 5))
    plt.barh(top_words, top_values)
    plt.title(f"Most important words for class: {label}")
    plt.xlabel("Coefficient value")
    plt.ylabel("Words")
    plt.show()

In [ ]:
# 10. MOST IMPORTANT WORDS - SVM

svm_model = models["SVM"]
coef = svm_model.coef_

for i, label in enumerate(label_names):
    top_idx = np.argsort(coef[i])[-15:]
    top_words = feature_names[top_idx]
    top_values = coef[i][top_idx]

    plt.figure(figsize=(8, 5))
    plt.barh(top_words, top_values)
    plt.title(f"Most important words - SVM: {label}")
    plt.xlabel("Coefficient value")
    plt.ylabel("Words")
    plt.show()

In [ ]:
# 11. MOST IMPORTANT WORDS - XGBoost

xgb_model = models["XGBoost"]

importances = xgb_model.feature_importances_

top_idx = np.argsort(importances)[-20:]

top_words = feature_names[top_idx]
top_values = importances[top_idx]

plt.figure(figsize=(8, 6))
plt.barh(top_words, top_values)
plt.title("Most important features - XGBoost")
plt.xlabel("Feature importance")
plt.ylabel("Words")
plt.show()